# Lab 0 Task 0.1 — CIFAR-10 CNN Experiments
This notebook implements a simple CNN for CIFAR-10 classification and compares different optimizers and activation functions as required for Task 0.1 of Lab 0.

## 1. Setup & Imports

In [16]:
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 2. Hyperparameters

In [17]:
BATCH_SIZE = 64
NUM_EPOCHS = 10

## 3. Data Loading

In [18]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_set = CIFAR10(root="../data", train=True,  download=True, transform=transform)
test_set  = CIFAR10(root="../data", train=False, download=True, transform=transform)

classes: list[str] = train_set.classes
print("Classes:", classes)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## 4. Model Definition

The `activation` argument lets us swap between `LeakyReLU` and `Tanh` without rewriting the class.

In [19]:
class SimpleCNN(nn.Module):
    """
    A simple CNN for CIFAR-10 classification.

    Architecture:
        Conv Block 1 : Conv2d(3  → 32)  → act → MaxPool2d  (32×32 → 16×16)
        Conv Block 2 : Conv2d(32 → 64)  → act → MaxPool2d  (16×16 →  8×8)
        Conv Block 3 : Conv2d(64 → 128) → act → MaxPool2d  ( 8×8  →  4×4)
        FC 1         : Linear(128*4*4 → 256) → act
        FC 2         : Linear(256 → num_classes)
    """

    def __init__(self, num_classes: int = 10, activation: nn.Module = nn.LeakyReLU()) -> None:
        super().__init__()

        self.conv1 = nn.Conv2d(3,  32,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.act  = activation
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = self.pool(self.act(self.conv3(x)))
        x = torch.flatten(x, start_dim=1)
        x = self.act(self.fc1(x))
        return self.fc2(x)

## 5. Training & Evaluation Helpers

`train_one_epoch` and `validate` each handle a single pass through their respective loader.  
`fit` orchestrates the loop, tracks history, and restores the best checkpoint.

In [20]:
criterion = nn.CrossEntropyLoss()


def train(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
) -> tuple[float, float]:
    """Run one full pass over `loader` in training mode.

    Returns
    -------
    train_loss : float  – mean cross-entropy loss over all samples
    train_acc  : float  – accuracy in percent
    """
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        correct      += (outputs.argmax(dim=1) == labels).sum().item()
        total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def validate(
    model: nn.Module,
    loader: DataLoader,
) -> tuple[float, float]:
    """Run one full pass over `loader` in evaluation mode.

    Returns
    -------
    val_loss : float  – mean cross-entropy loss over all samples
    val_acc  : float  – accuracy in percent
    """
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            correct      += (outputs.argmax(dim=1) == labels).sum().item()
            total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def fit(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int = NUM_EPOCHS,
) -> dict[str, list[float]]:
    """Train `model` for `num_epochs`, validating after every epoch.

    Saves the weights that achieved the best validation accuracy and
    restores them at the end.

    Returns
    -------
    history : dict with keys 'train_loss', 'val_loss', 'train_acc', 'val_acc'
    """
    best_val_acc = 0.0
    best_state   = None
    history: dict[str, list[float]] = {
        "train_loss": [], "val_loss": [],
        "train_acc":  [], "val_acc":  [],
    }

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train(model, train_loader, optimizer)
        val_loss, val_acc = validate(model, val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:>{len(str(num_epochs))}}/{num_epochs} | "
            f"train loss {train_loss:.4f}, train acc {train_acc:.2f}% | "
            f"val loss {val_loss:.4f}, val acc {val_acc:.2f}%"
        )

    # Restore best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\nRestored best weights (val acc {best_val_acc:.2f}%)")

    return history

## 6. Experiment A – SGD + LeakyReLU
*(The original baseline from the script)*

In [ ]:
model_a    = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_a = optim.SGD(model_a.parameters(), lr=1e-2)

print("=== Experiment A: SGD + LeakyReLU ===")
history_a = fit(model_a, optimizer_a, train_loader, test_loader)

=== Experiment A: SGD + LeakyReLU ===
Epoch  1/10 | train loss 2.2440, train acc 18.16% | val loss 2.0743, val acc 24.51%


## 7. Experiment B – Adam + LeakyReLU
*(Task: swap SGD for Adam and report accuracy)*

In [14]:
model_b     = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_b = optim.Adam(model_b.parameters(), lr=1e-3)

print("=== Experiment B: Adam + LeakyReLU ===")
history_b = fit(model_b, optimizer_b, train_loader, test_loader)

=== Experiment B: Adam + LeakyReLU ===
Epoch  1/10 | train loss 1.3503, train acc 50.79% | val loss 1.0336, val acc 63.29%
Epoch  2/10 | train loss 0.9077, train acc 67.89% | val loss 0.8889, val acc 68.57%
Epoch  3/10 | train loss 0.7222, train acc 74.56% | val loss 0.7788, val acc 73.29%
Epoch  4/10 | train loss 0.5803, train acc 79.61% | val loss 0.7647, val acc 74.31%
Epoch  5/10 | train loss 0.4625, train acc 83.76% | val loss 0.7625, val acc 75.38%
Epoch  6/10 | train loss 0.3502, train acc 87.75% | val loss 0.7883, val acc 75.24%
Epoch  7/10 | train loss 0.2614, train acc 90.78% | val loss 0.8440, val acc 75.90%
Epoch  8/10 | train loss 0.1829, train acc 93.60% | val loss 0.9534, val acc 75.56%
Epoch  9/10 | train loss 0.1334, train acc 95.33% | val loss 1.1159, val acc 75.16%
Epoch 10/10 | train loss 0.1080, train acc 96.20% | val loss 1.2778, val acc 75.33%

Restored best weights (val acc 75.90%)

Adam + LeakyReLU final test accuracy: 75.33%


## 8. Experiment C – Adam + Tanh
*(Task: swap LeakyReLU for Tanh and report accuracy)*

In [15]:
model_c     = SimpleCNN(num_classes=len(classes), activation=nn.Tanh()).to(device)
optimizer_c = optim.Adam(model_c.parameters(), lr=1e-3)

print("=== Experiment C: Adam + Tanh ===")
history_c = fit(model_c, optimizer_c, train_loader, test_loader)

=== Experiment C: Adam + Tanh ===
Epoch  1/10 | train loss 1.2552, train acc 55.14% | val loss 0.9923, val acc 65.27%
Epoch  2/10 | train loss 0.8787, train acc 69.44% | val loss 0.8716, val acc 69.72%
Epoch  3/10 | train loss 0.7082, train acc 75.26% | val loss 0.8147, val acc 71.58%
Epoch  4/10 | train loss 0.5819, train acc 80.13% | val loss 0.8288, val acc 71.69%
Epoch  5/10 | train loss 0.4761, train acc 83.38% | val loss 0.7825, val acc 73.83%
Epoch  6/10 | train loss 0.3751, train acc 87.23% | val loss 0.8166, val acc 73.74%
Epoch  7/10 | train loss 0.2844, train acc 90.47% | val loss 0.8977, val acc 73.26%
Epoch  8/10 | train loss 0.2193, train acc 92.79% | val loss 0.9371, val acc 72.88%
Epoch  9/10 | train loss 0.1606, train acc 94.92% | val loss 1.0098, val acc 72.61%
Epoch 10/10 | train loss 0.1316, train acc 95.78% | val loss 1.0371, val acc 72.88%

Restored best weights (val acc 73.83%)

Adam + Tanh final test accuracy: 72.88%
